# 🔧 QoS Data Preparation Pipeline
### KNN Imputer + RobustScaler
---

# Data Preparation

# DSO1 +DSO2 — Surveillance Réseau Temps Réel
### Jupyter Lab / Jupyter Notebook / VS Code
**Agent 1** : capture réseau (54 features) · **Agent 2** : classification & alertes  
**Ordre :** Cellule 1 → Cellule 2 → Cellule 3 (START) → Cellule 4 (test optionnel)


# DSO3 :RISK PREDECTION 

## 📦 Cellule 1 — Installation des dépendances

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 1 — Installation
# Exécuter une seule fois, puis redémarrer le kernel si nécessaire
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

packages = ['ping3', 'icmplib', 'psutil', 'requests', 'ipywidgets', 'jupyterlab']
for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f'{status} {pkg}')

print()
print('⚠️  Si premier lancement → redémarrer le kernel (Kernel > Restart)')
print('   puis exécuter les cellules 1 → 2 → 3')


╔═══════════════════════════════════════════════════════════════════╗
║  🎯  DSO 3 — Prédiction des Risques Réseau (temps réel)            ║
║  Prédiction #39    · 19:04:12                                      ║
║  Risque : 🟢 LOW                                                      ║
╚═══════════════════════════════════════════════════════════════════╝

┌──────────────────────────────────────────────────────────────────┐
│  📊  Score de risque courant                                      │
└──────────────────────────────────────────────────────────────────┘

  [█████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 27.4/100

  Composante                        Score
  ────────────────────────────────────────
  Séquentiel (LSTM/stat)             46.5/100
  Tabulaire (XGB/heur.)               4.1/100
  Anomalie (Isolation F.)           ✅ NON
  Score anomalie                   0.0000

┌──────────────────────────────────────────────────────────────────┐
│  🔭  Projection multi-horizon           

## ⚙️ Cellule 2 — Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 2 — CONFIGURATION
# Seuils selon Cisco QoS Design Guide + ITU-T G.114 + 3GPP TS 36
# ═══════════════════════════════════════════════════════════════
import os

# ── Réseau
TARGET_HOST  = '8.8.8.8'
TARGET_HOST2 = '1.1.1.1'
PING_COUNT   = 5
MAX_HOPS     = 10
INTERVAL_SEC = 10

# ── Chemins CSV
OUTPUT_DIR  = os.path.join(os.getcwd(), 'qos_output')
CSV_LIVE    = os.path.join(OUTPUT_DIR, 'monitoring_live.csv')
CSV_ALERTS  = os.path.join(OUTPUT_DIR, 'alerts_log.csv')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Webhook
WEBHOOK_URL = None

# ── Seuils Cisco / ITU-T / 3GPP
THRESHOLDS = {
    # Latence — ITU-T G.114
    'latency_ms':                (150,   200,   False),
    'mean_latency_ms':           (150,   200,   False),
    'max_latency_ms':            (200,   400,   False),
    'std_latency_ms':            (20,    50,    False),
    # Jitter — ITU-T G.114
    'jitter_ms':                 (30,    50,    False),
    'latency_spread':            (20,    50,    False),
    'latency_trend':             (50,    100,   False),
    # Perte paquets — Cisco QoS
    'packet_loss_rate_pct':      (1.0,   5.0,   False),
    'risk_score':                (0.4,   0.6,   False),
    'instability_score':         (5.0,   15.0,  False),
    # Bande passante — Cisco (seuils utilisation réseau)
    'bandwidth_utilization_pct': (50.0,  75.0,  False),
    'network_load':              (0.5,   0.75,  False),
    # Charge système
    'queue_length':              (70,    90,    False),
    'buffer_occupancy_pct':      (75,    90,    False),
    'prb_utilization_proxy':     (50.0,  75.0,  False),
    # Hops
    'hops_mean':                 (50,    100,   False),
    'hops_max':                  (150,   300,   False),
    'hops_std':                  (30,    60,    False),
    'hops_range':                (100,   200,   False),
    # ── throughput et available_bandwidth SUPPRIMÉS
    #    (psutil mesure le trafic local, pas la vraie bande passante)
    # MOS — ITU-T P.800
    'mos_proxy':                 (3.6,   3.1,   True),
    # Signal LTE — 3GPP TS 36.133
    'rsrp_estimated':            (-80,   -100,  True),
    'sinr_estimated':            (0,     -3,    True),
    'cqi_estimated':             (7,     4,     True),
    # Efficacité bande passante
    'bandwidth_efficiency':      (0.3,   0.1,   True),
}

BINARY_FLAGS = {
    'spike':               'Pic de latence (ITU-T G.114)',
    'ho_failure_proxy':    'Échec handover (3GPP TS 36)',
    'coverage_hole_proxy': 'Zone couverture faible (RSRP < -100 dBm)',
    'performance_degraded':'Dégradation performance (Cisco QoS)',
}

print('✅ Configuration chargée — seuils Cisco/ITU-T/3GPP')
print(f'   Cible      : {TARGET_HOST} / {TARGET_HOST2}')
print(f'   Intervalle : {INTERVAL_SEC}s')
print(f'   CSV live   : {CSV_LIVE}')
print(f'   CSV alertes: {CSV_ALERTS}')
print(f'   Webhook    : {WEBHOOK_URL or "désactivé"}')
print(f'   Seuils     : {len(THRESHOLDS)} features continues + {len(BINARY_FLAGS)} binaires')

✅ Configuration chargée — seuils Cisco/ITU-T/3GPP
   Cible      : 8.8.8.8 / 1.1.1.1
   Intervalle : 10s
   CSV live   : c:\Users\ayoub\Downloads\qos_output\monitoring_live.csv
   CSV alertes: c:\Users\ayoub\Downloads\qos_output\alerts_log.csv
   Webhook    : désactivé
   Seuils     : 24 features continues + 4 binaires


## 🚀 Cellule 3 — Agent 1 + Agent 2 + Dashboard temps réel

In [ ]:
# ── Vérification des dépendances ───────────────────────────────
_required = ['CSV_LIVE', 'CSV_ALERTS', 'TARGET_HOST', 'TARGET_HOST2',
             'PING_COUNT', 'MAX_HOPS', 'INTERVAL_SEC',
             'THRESHOLDS', 'BINARY_FLAGS', 'WEBHOOK_URL']

_missing = [v for v in _required if v not in dir()]
if _missing:
    raise RuntimeError(
        f"❌ Variables manquantes : {_missing}\n"
        f"   → Exécute d'abord la Cellule 2 (Configuration) !"
    )
print('✅ Dépendances OK — chargement des agents...')

✅ Dépendances OK — chargement des agents...


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 3 — AGENT 1 + AGENT 2 + DASHBOARD
# Jupyter Lab / Notebook / VS Code — sans widgets
# ═══════════════════════════════════════════════════════════════
import psutil, time, os, csv, json as _json, math, statistics
import requests, threading, queue
from ping3 import ping
from icmplib import traceroute
from datetime import datetime
from IPython.display import clear_output

# ────────────────────────────────────────────────────────────────
# ÉTAT PARTAGÉ + FILE INTER-AGENTS
# ────────────────────────────────────────────────────────────────
state = {
    'running':       False,
    'total':         0,
    'alerts_count':  0,
    'last_row':      None,
    'last_analysis': None,
    'a1_log':        [],   # historique logs Agent 1
    'a2_log':        [],   # historique logs Agent 2
    'api_status':    '—',
}
data_queue    = queue.Queue(maxsize=5)
alert_history = []

# ────────────────────────────────────────────────────────────────
# AGENT 1 — CAPTURE RÉSEAU (54 features)
# ────────────────────────────────────────────────────────────────
def _save_csv(row, path):
    exists = os.path.isfile(path)
    with open(path, 'a', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=row.keys(), delimiter=';')
        if not exists:
            w.writeheader()
        w.writerow(row)

def _a1_log(msg):
    ts = datetime.now().strftime('%H:%M:%S')
    state['a1_log'].insert(0, f'[{ts}] {msg}')
    state['a1_log'] = state['a1_log'][:20]

def agent1_capture():
    ts  = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    now = datetime.now()
    row = {'timestamp': ts}

    # 1. LATENCE
    pings, host = [], TARGET_HOST
    for _ in range(PING_COUNT):
        try:
            r = ping(TARGET_HOST, timeout=2, unit='ms')
            if r: pings.append(r)
        except: pass
    if not pings:
        host = TARGET_HOST2
        for _ in range(PING_COUNT):
            try:
                r = ping(TARGET_HOST2, timeout=2, unit='ms')
                if r: pings.append(r)
            except: pass

    row['latency_ms']           = round(sum(pings)/len(pings), 4) if pings else -1
    row['mean_latency_ms']      = row['latency_ms']
    row['min_latency_ms']       = round(min(pings), 4)            if pings else -1
    row['max_latency_ms']       = round(max(pings), 4)            if pings else -1
    row['std_latency_ms']       = round(statistics.stdev(pings), 4) if len(pings)>1 else 0.0
    row['jitter_ms']            = round(row['max_latency_ms'] - row['min_latency_ms'], 4) if pings else -1
    row['latency_spread']       = row['std_latency_ms']
    row['packet_loss_rate_pct'] = round((PING_COUNT - len(pings)) / PING_COUNT * 100, 2)
    row['spike']                = 1 if row['latency_ms'] > 200 else 0
    prev = float(state['last_row']['latency_ms']) if state['last_row'] else row['latency_ms']
    row['latency_trend']        = round(row['latency_ms'] - prev, 4)

    # 2. BANDE PASSANTE
    net1 = psutil.net_io_counters(); time.sleep(1); net2 = psutil.net_io_counters()
    s = (net2.bytes_sent - net1.bytes_sent) * 8
    r = (net2.bytes_recv - net1.bytes_recv) * 8
    row['throughput_mbps']            = round(r / 1e6, 4)
    row['available_bandwidth_mbps']   = round((s + r) / 1e6, 4)
    row['bandwidth_utilization_pct']  = round(min(r / 1e8 * 100, 100), 4)
    row['bandwidth_efficiency']       = round(r / max(s + r, 1), 4)
    row['network_load']               = round(row['bandwidth_utilization_pct'] / 100, 4)

    # 3. QUALITÉ RÉSEAU
    drops = (net2.dropin + net2.dropout) - (net1.dropin + net1.dropout)
    pkts  = max(net2.packets_recv - net1.packets_recv, 1)
    row['packet_loss_rate_pct'] = round(drops / pkts * 100, 4)
    row['instability_score']    = round(row['jitter_ms'] * row['packet_loss_rate_pct'] / 100, 4)
    row['risk_score']           = round((row['latency_ms']/300 + row['packet_loss_rate_pct']/100) / 2, 4)

    # 4. CHARGE SYSTÈME
    row['buffer_occupancy_pct']  = round(psutil.virtual_memory().percent, 4)
    row['queue_length']          = round(psutil.cpu_percent(interval=0.2), 4)
    row['congestion_level']      = 0
    row['prb_utilization_proxy'] = row['bandwidth_utilization_pct']

    # 5. HOPS
    hl = []
    try:
        hops = traceroute(host, max_hops=MAX_HOPS, timeout=1)
        hl = [h.avg_rtt for h in hops if h.avg_rtt > 0]
    except: pass
    for i in range(1, 11):
        row[f'hop_{i}'] = round(hl[i-1], 4) if i <= len(hl) else 0.0
    row['hops_mean']  = round(sum(hl)/len(hl), 4) if hl else 0.0
    row['hops_max']   = round(max(hl), 4)          if hl else 0.0
    row['hops_min']   = round(min(hl), 4)          if hl else 0.0
    row['hops_std']   = round(statistics.stdev(hl), 4) if len(hl) > 1 else 0.0
    row['hops_range'] = round(row['hops_max'] - row['hops_min'], 4)

    # 6. SIGNAL ESTIMÉ
    row['rsrp_estimated'] = round(-70 - row['latency_ms'] * 0.1, 4)
    row['sinr_estimated'] = round(20 - row['packet_loss_rate_pct'] * 2, 4)
    row['cqi_estimated']  = max(1, min(15, int(15 - row['packet_loss_rate_pct'] / 7)))
    row['mos_proxy']      = round(max(1, 4.5 - row['packet_loss_rate_pct']*0.1 - row['latency_ms']*0.005), 4)

    # 7. FLAGS BINAIRES
    row['ho_failure_proxy']     = 1 if row['packet_loss_rate_pct'] > 5 else 0
    row['coverage_hole_proxy']  = 1 if row['rsrp_estimated'] < -100    else 0
    row['performance_degraded'] = 1 if (row['latency_ms'] > 150 or row['packet_loss_rate_pct'] > 3) else 0

    # 8. CATÉGORIE RSRP
    rsrp = row['rsrp_estimated']
    row['rsrp_category'] = ('Bon'          if rsrp >= -80  else
                            'Mauvais'      if rsrp >= -90  else
                            'Faible'       if rsrp >= -100 else
                            'Très mauvais')

    # 9. TEMPOREL
    row['hour']                   = now.hour
    row['minute']                 = now.minute
    row['dayofweek']              = now.weekday()
    row['hour_sin']               = round(math.sin(2*math.pi*now.hour/24), 4)
    row['hour_cos']               = round(math.cos(2*math.pi*now.hour/24), 4)
    row['minute_sin']             = round(math.sin(2*math.pi*now.minute/60), 4)
    row['minute_cos']             = round(math.cos(2*math.pi*now.minute/60), 4)
    row['dayofweek_sin']          = round(math.sin(2*math.pi*now.weekday()/7), 4)
    row['dayofweek_cos']          = round(math.cos(2*math.pi*now.weekday()/7), 4)
    row['peak_offpeak_indicator'] = 1 if 8 <= now.hour <= 22 else 0
    return row

def agent1_loop():
    _a1_log('démarré')
    while state['running']:
        state['total'] += 1
        n = state['total']
        _a1_log(f'⏱  capture #{n} en cours...')
        try:
            row = agent1_capture()
            state['last_row'] = row
            _save_csv(row, CSV_LIVE)
            try: data_queue.put_nowait(row)
            except: pass
            _a1_log(f'✅ #{n} — {len(row)} features — latence {row["latency_ms"]} ms')
        except Exception as e:
            _a1_log(f'❌ #{n} erreur: {e}')
        for _ in range(INTERVAL_SEC):
            if not state['running']: break
            time.sleep(1)
    _a1_log(f'⏹  arrêté après {state["total"]} captures')

# ────────────────────────────────────────────────────────────────
# AGENT 2 — CLASSIFICATION & ALERTES
# ────────────────────────────────────────────────────────────────
def _a2_log(msg):
    ts = datetime.now().strftime('%H:%M:%S')
    state['a2_log'].insert(0, f'[{ts}] {msg}')
    state['a2_log'] = state['a2_log'][:20]

def agent2_classify(row):
    alerts = []
    for metric, (warn, crit, inv) in THRESHOLDS.items():
        val = row.get(metric)
        if val is None or val == -1: continue
        v = float(val)
        level = ('CRITICAL' if (v <= crit if inv else v >= crit) else
                 'WARNING'  if (v <= warn if inv else v >= warn) else 'OK')
        if level != 'OK':
            alerts.append({
                'metric':    metric,
                'value':     round(v, 4),
                'threshold': crit if level == 'CRITICAL' else warn,
                'level':     level,
            })
    for metric, desc in BINARY_FLAGS.items():
        if int(float(row.get(metric, 0))) == 1:
            alerts.append({'metric': metric, 'value': 1,
                           'description': desc, 'level': 'CRITICAL'})
    has_crit = any(a['level'] == 'CRITICAL' for a in alerts)
    return {
        'alerts':       alerts,
        'severity':     'CRITICAL' if has_crit else ('WARNING' if alerts else 'OK'),
        'has_critical': has_crit,
        'rsrp_class':   row.get('rsrp_category', '?'),
        'n_critical':   sum(1 for a in alerts if a['level'] == 'CRITICAL'),
        'n_warning':    sum(1 for a in alerts if a['level'] == 'WARNING'),
    }

def build_payload(row, analysis):
    return {
        'agent':     'QoS-Agent2-Classifier',
        'timestamp':  row.get('timestamp', ''),
        'target':     TARGET_HOST,
        'severity':   analysis['severity'],
        'rsrp_class': analysis['rsrp_class'],
        'n_critical': analysis['n_critical'],
        'n_warning':  analysis['n_warning'],
        'alerts':     analysis['alerts'],
        'metrics':    {k: row.get(k) for k in [
            'latency_ms','mean_latency_ms','max_latency_ms','jitter_ms','latency_trend',
            'packet_loss_rate_pct','risk_score','instability_score',
            'throughput_mbps','bandwidth_utilization_pct','mos_proxy',
            'rsrp_estimated','sinr_estimated','cqi_estimated','hops_mean',
            'buffer_occupancy_pct','queue_length','spike','ho_failure_proxy',
            'coverage_hole_proxy','performance_degraded','rsrp_category',
        ]}
    }

def send_webhook(payload):
    if not WEBHOOK_URL: return 'désactivé'
    try:
        r = requests.post(WEBHOOK_URL, json=payload, timeout=5,
                          headers={'Content-Type': 'application/json'})
        r.raise_for_status()
        return f'HTTP {r.status_code} ✅'
    except Exception as e:
        return f'Erreur: {e}'

def agent2_loop():
    _a2_log('démarré — en écoute')
    while state['running']:
        try: row = data_queue.get(timeout=2)
        except: continue
        try:
            analysis = agent2_classify(row)
            state['last_analysis'] = analysis

            # ── CORRECTIF : transmettre (row, analysis) à DSO3
            try:
                dso3_input_queue.put_nowait((row, analysis))
            except queue.Full:
                pass

            if analysis['severity'] != 'OK':
                state['alerts_count'] += 1
                # Sauvegarde CSV alertes
                exists = os.path.isfile(CSV_ALERTS)
                with open(CSV_ALERTS, 'a', newline='', encoding='utf-8') as f:
                    fw = csv.DictWriter(f, delimiter=';', fieldnames=[
                        'timestamp','severity','rsrp_class',
                        'metric','value','threshold','level','description'])
                    if not exists: fw.writeheader()
                    for a in analysis['alerts']:
                        fw.writerow({
                            'timestamp':   row['timestamp'],
                            'severity':    analysis['severity'],
                            'rsrp_class':  analysis['rsrp_class'],
                            'metric':      a.get('metric',''),
                            'value':       a.get('value',''),
                            'threshold':   a.get('threshold',''),
                            'level':       a['level'],
                            'description': a.get('description', a.get('metric','')),
                        })
                if WEBHOOK_URL:
                    state['api_status'] = send_webhook(build_payload(row, analysis))
                _a2_log(f'🚨 {analysis["severity"]} — {analysis["n_critical"]}C {analysis["n_warning"]}W alertes')
            else:
                _a2_log(f'✅ OK — RSRP: {analysis["rsrp_class"]}')

        except Exception as e:
            _a2_log(f'❌ erreur: {e}')
    _a2_log(f'⏹  arrêté — {state["alerts_count"]} alertes déclenchées')

# ────────────────────────────────────────────────────────────────
# DASHBOARD — affichage séparé Agent 1 / Agent 2
# ────────────────────────────────────────────────────────────────
GROUPS_A1 = {
    '⏱️  Latence':       ['latency_ms','mean_latency_ms','min_latency_ms','max_latency_ms',
                          'std_latency_ms','jitter_ms','latency_spread','latency_trend','spike'],
    '📶 Bande passante': ['throughput_mbps','available_bandwidth_mbps',
                          'bandwidth_utilization_pct','bandwidth_efficiency','network_load'],
    '📉 Qualité réseau': ['packet_loss_rate_pct','instability_score','risk_score'],
    '🔧 Charge système': ['queue_length','buffer_occupancy_pct','prb_utilization_proxy'],
    '📡 Signal estimé':  ['rsrp_estimated','sinr_estimated','cqi_estimated','mos_proxy','rsrp_category'],
    '🛤️  Hops':          ['hops_mean','hops_max','hops_min','hops_std','hops_range',
                          'hop_1','hop_2','hop_3','hop_4','hop_5',
                          'hop_6','hop_7','hop_8','hop_9','hop_10'],
    '🕐 Temporel':       ['hour','minute','dayofweek','peak_offpeak_indicator',
                          'hour_sin','hour_cos','minute_sin','minute_cos',
                          'dayofweek_sin','dayofweek_cos'],
    '🚨 Flags binaires': ['spike','ho_failure_proxy','coverage_hole_proxy','performance_degraded'],
}

def _flag_note(metric, val):
    t = THRESHOLDS.get(metric)
    if not t: return ''
    warn, crit, inv = t
    v = float(val)
    if inv:
        if v <= crit: return '  🔴 CRITIQUE'
        if v <= warn: return '  🟡 AVERT.'
    else:
        if v >= crit: return '  🔴 CRITIQUE'
        if v >= warn: return '  🟡 AVERT.'
    return ''

def render():
    row = state.get('last_row')
    ana = state.get('last_analysis')
    n   = state.get('total', 0)
    clear_output(wait=True)

    # ══════════════════════════════════════════════════════
    # HEADER GLOBAL
    # ══════════════════════════════════════════════════════
    sev   = ana['severity'] if ana else '—'
    emoji = '🔴' if sev=='CRITICAL' else ('🟡' if sev=='WARNING' else '🟢')
    print('╔' + '═'*63 + '╗')
    print(f'║  🛰️  QoS Real-Time Monitor — DSO1' + ' '*29 + '║')
    print(f'║  Capture #{n:<6} · {datetime.now().strftime("%H:%M:%S")}' + ' '*36 + '║')
    print(f'║  Statut : {emoji} {sev:<10} · Alertes totales : {state["alerts_count"]:<6}' + ' '*14 + '║')
    print('╚' + '═'*63 + '╝')

    # ══════════════════════════════════════════════════════
    # AGENT 1 — CAPTURE
    # ══════════════════════════════════════════════════════
    print()
    print('┌─────────────────────────────────────────────────────────────┐')
    print('│  🤖  AGENT 1 — Capture réseau (54 features)                 │')
    print('└─────────────────────────────────────────────────────────────┘')

    if not row:
        print('  ⏳ En attente de la première capture...')
    else:
        for grp, feats in GROUPS_A1.items():
            visible = [f for f in feats if f in row]
            if not visible: continue
            print(f'\n  {grp}')
            print(f'  {"─"*58}')
            print(f'  {"Feature":<36} {"Valeur":>10}  {"Statut"}')
            print(f'  {"─"*58}')
            for feat in visible:
                val  = row[feat]
                note = ''
                if feat in BINARY_FLAGS:
                    note = '🔴 ACTIF' if int(float(val))==1 else '✅ OK'
                elif feat in THRESHOLDS:
                    n_raw = _flag_note(feat, val)
                    note  = n_raw.strip() if n_raw else '✅ OK'
                print(f'  {feat:<36} {str(val):>10}  {note}')

    # Logs Agent 1
    print(f'\n  📋 Logs Agent 1 :')
    for line in state['a1_log'][:5]:
        print(f'     {line}')

    print(f'\n  💾 CSV live → {CSV_LIVE}')

    # ══════════════════════════════════════════════════════
    # AGENT 2 — CLASSIFICATION
    # ══════════════════════════════════════════════════════
    print()
    print('┌─────────────────────────────────────────────────────────────┐')
    print('│  🔍  AGENT 2 — Classification & Alertes                     │')
    print('└─────────────────────────────────────────────────────────────┘')

    if not ana:
        print('  ⏳ En attente de la première classification...')
    else:
        sev_a2   = ana['severity']
        emoji_a2 = '🔴' if sev_a2=='CRITICAL' else ('🟡' if sev_a2=='WARNING' else '🟢')
        print(f'\n  Sévérité   : {emoji_a2} {sev_a2}')
        print(f'  RSRP class : {ana["rsrp_class"]}')
        print(f'  Critiques  : {ana["n_critical"]}   Avertissements : {ana["n_warning"]}')

        if ana['alerts']:
            print(f'\n  {"─"*58}')
            print(f'  {"Metric":<32} {"Valeur":>8}  {"Seuil":>8}  {"Niveau"}')
            print(f'  {"─"*58}')
            for a in ana['alerts']:
                icon  = '🔴' if a['level']=='CRITICAL' else '🟡'
                desc  = a.get('description', a['metric'])
                val   = str(a.get('value',''))
                thr   = str(a.get('threshold',''))
                print(f'  {icon} {desc:<30} {val:>8}  {thr:>8}  {a["level"]}')
        else:
            print('\n  ✅ Aucune alerte — tous les indicateurs dans les seuils')

    # Logs Agent 2
    print(f'\n  📋 Logs Agent 2 :')
    for line in state['a2_log'][:5]:
        print(f'     {line}')

    print(f'\n  💾 CSV alertes → {CSV_ALERTS}')
    if WEBHOOK_URL:
        print(f'  📤 Webhook     → {state.get("api_status","—")}')

    # ══════════════════════════════════════════════════════
    # JOURNAL ALERTES (5 dernières)
    # ══════════════════════════════════════════════════════
    if alert_history:
        print()
        print('┌─────────────────────────────────────────────────────────────┐')
        print('│  🚨  Journal alertes                                         │')
        print('└─────────────────────────────────────────────────────────────┘')
        for line in alert_history[:5]:
            print(f'  {line}')
        if len(alert_history) > 5:
            print(f'  ... +{len(alert_history)-5} alertes dans le journal complet')

    print()
    print('  ▶ on_start()  ■ on_stop()  ↺ on_reset()  🔄 render()')

    # Mise à jour journal alertes
    if ana and ana['alerts'] and row:
        ts  = row['timestamp']
        sev = ana['severity']
        for a in ana['alerts']:
            desc  = a.get('description', f'{a["metric"]}={a["value"]} seuil={a.get("threshold","")}')
            entry = f'[{ts}] [{sev}] {desc}'
            if entry not in alert_history:
                alert_history.insert(0, entry)
        alert_history[:] = alert_history[:200]

def dashboard_loop():
    while state['running']:
        try: render()
        except: pass
        time.sleep(INTERVAL_SEC)

# ────────────────────────────────────────────────────────────────
# COMMANDES
# ────────────────────────────────────────────────────────────────
def on_start():
    if state['running']:
        print('⚠️  Déjà en cours — on_stop() pour arrêter')
        return
    state.update({
        'running': True, 'total': 0, 'alerts_count': 0,
        'last_row': None, 'last_analysis': None,
        'a1_log': [], 'a2_log': [], 'api_status': '—',
    })
    alert_history.clear()
    threading.Thread(target=agent1_loop,    daemon=True).start()
    threading.Thread(target=agent2_loop,    daemon=True).start()
    threading.Thread(target=dashboard_loop, daemon=True).start()
    print(f'🟢 Agents démarrés — intervalle {INTERVAL_SEC}s — cible {TARGET_HOST}')
    print('   Le dashboard se rafraîchit automatiquement.')
    print('   Pour affichage immédiat → render()')
'''
def on_stop():
    state['running'] = False
    print(f'⏹  Arrêté après {state["total"]} captures · {state["alerts_count"]} alertes')

def on_reset():
    state['running'] = False
    alert_history.clear()
    state.update({'total':0,'alerts_count':0,'last_row':None,'last_analysis':None,
                  'a1_log':[],'a2_log':[],'api_status':'—'})
    print('↺ Réinitialisé — prêt')
'''
print('✅ Agents définis')
print('▶  on_start()  →  démarrer')
print('■  on_stop()   →  arrêter')
print('↺  on_reset()  →  réinitialiser')
print('🔄  render()   →  affichage immédiat')

✅ Agents définis
▶  on_start()  →  démarrer
■  on_stop()   →  arrêter
↺  on_reset()  →  réinitialiser
🔄  render()   →  affichage immédiat


In [ ]:
on_start()

╔═══════════════════════════════════════════════════════════════╗
║  🛰️  QoS Real-Time Monitor — DSO1                             ║
║  Capture #1      · 19:04:19                                    ║
║  Statut : 🟢 —          · Alertes totales : 0                   ║
╚═══════════════════════════════════════════════════════════════╝

┌─────────────────────────────────────────────────────────────┐
│  🤖  AGENT 1 — Capture réseau (54 features)                 │
└─────────────────────────────────────────────────────────────┘
  ⏳ En attente de la première capture...

  📋 Logs Agent 1 :
     [19:04:19] ⏱  capture #1 en cours...
     [19:04:19] démarré

  💾 CSV live → c:\Users\ayoub\Downloads\qos_output\monitoring_live.csv

┌─────────────────────────────────────────────────────────────┐
│  🔍  AGENT 2 — Classification & Alertes                     │
└─────────────────────────────────────────────────────────────┘
  ⏳ En attente de la première classification...

  📋 Logs Agent 2 :
     [19:04:19] d

In [ ]:
#on_stop()

## 📦 1. Installation des dépendances

In [ ]:
!pip install scikit-learn joblib -q
print('✅ Dépendances installées')

## 📁 2. Upload du fichier CSV

In [ ]:
import pandas as pd
import requests
from io import StringIO

file_id = '1H3__N0mCX9OdI_xWqRp3trgx-9OUPmjn'
url = f'https://drive.google.com/uc?export=download&id={file_id}'
response = requests.get(url)
df = pd.read_csv(StringIO(response.text), sep=';')

print(f'✅ Fichier chargé depuis Google Drive')
print(f'📊 Shape : {df.shape}')

## 📚 3. Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings
import os
warnings.filterwarnings('ignore')

from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler, LabelEncoder
import joblib

print('✅ Imports OK')

## 📊 4. Chargement & Analyse initiale

In [ ]:
print('=' * 60)
print(f'  Lignes     : {df.shape[0]}')
print(f'  Colonnes   : {df.shape[1]}')
print('=' * 60)
print(f'📌 Colonnes numériques  : {df.select_dtypes(include="number").shape[1]}')
print(f'📌 Colonnes texte       : {df.select_dtypes(include="object").shape[1]}')
print(f'📌 Valeurs manquantes   : {df.isnull().sum().sum()}')

df.head(3)

## 🗂️ 5. Séparation des colonnes par rôle

In [ ]:
# Colonnes à supprimer (non pertinentes)
cols_to_drop = ['timestamp', 'congestion_level']  # congestion_level = 0 partout

# Cible (target)
target_col = 'rsrp_category'

# Colonnes binaires (pas besoin de scaler)
binary_cols = ['spike', 'ho_failure_proxy', 'coverage_hole_proxy', 'peak_offpeak_indicator']

# Toutes les features
feature_cols = [c for c in df.columns if c not in cols_to_drop + [target_col]]
numeric_features = [c for c in feature_cols if df[c].dtype in ['float64', 'int64']]

print(f'📌 Colonnes supprimées : {cols_to_drop}')
print(f'📌 Target              : {target_col}')
print(f'📌 Features totales    : {len(feature_cols)}')
print(f'   • Continues        : {len([c for c in numeric_features if c not in binary_cols])}')
print(f'   • Binaires         : {len(binary_cols)}')

## 🎯 6. Encodage de la target

In [ ]:
print(df.columns.tolist()) 
# Détection automatique de la colonne target
target_col = None
for candidate in ['rsrp_category', 'rsrp_category_label', 'rsrp_cat', 'category']:
    if candidate in df.columns:
        target_col = candidate
        break

if target_col is None:
    print('❌ Colonne target non trouvée — colonnes disponibles :')
    print(df.columns.tolist())
else:
    print(f'✅ Target trouvée : {target_col}')
    le = LabelEncoder()
    y = le.fit_transform(df[target_col])
    label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))

    print('🎯 Mapping target :')
    for label, code in sorted(label_mapping.items(), key=lambda x: x[1]):
        count = (df[target_col] == label).sum()
        pct = count / len(df) * 100
        print(f'   {code} → {label:15s} : {count:5d} ({pct:.1f}%)')

    import matplotlib.pyplot as plt
    available = [k for k in ['Bon', 'Faible', 'Mauvais', 'Très mauvais'] if k in df[target_col].values]
    colors_map = {'Bon':'#2ecc71','Faible':'#f39c12','Mauvais':'#e74c3c','Très mauvais':'#8e44ad'}
    counts = [df[target_col].value_counts()[k] for k in available]
    colors = [colors_map[k] for k in available]

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(available, counts, color=colors, edgecolor='white', linewidth=1.5)
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                str(cnt), ha='center', fontsize=11, fontweight='bold')
    ax.set_title(f'Distribution de {target_col}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Nombre de samples')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

## 🔵 7. KNN Imputer (k=5, weights='distance')

In [ ]:
X = df[feature_cols].copy()
print(f'Shape X avant imputation : {X.shape}')
print(f'NaN avant               : {X.isnull().sum().sum()}')

knn_imputer = KNNImputer(n_neighbors=5, weights='distance')
X_imputed = knn_imputer.fit_transform(X)
X_imputed = pd.DataFrame(X_imputed, columns=feature_cols)

print(f'\n✅ Imputation terminée')
print(f'NaN après               : {X_imputed.isnull().sum().sum()}')

## 🔴 8. RobustScaler (IQR 10–90) — résistant aux outliers

In [ ]:
cols_to_scale = [c for c in feature_cols if c not in binary_cols]

scaler = RobustScaler(quantile_range=(10, 90))
X_scaled = X_imputed.copy()
X_scaled[cols_to_scale] = scaler.fit_transform(X_imputed[cols_to_scale])

print(f'✅ RobustScaler appliqué sur {len(cols_to_scale)} colonnes')
print(f'✅ Colonnes binaires conservées : {binary_cols}')
print(f'NaN final : {X_scaled.isnull().sum().sum()}')

# Visualisation avant/après sur 4 features clés
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
key_features = ['throughput_mbps', 'latency_ms', 'jitter_ms', 'rsrp_estimated']
colors_before = '#3498db'
colors_after  = '#e74c3c'

for i, feat in enumerate(key_features):
    axes[0, i].hist(X_imputed[feat], bins=40, color=colors_before, alpha=0.8, edgecolor='white')
    axes[0, i].set_title(f'{feat}\n(avant)', fontsize=9)
    axes[0, i].spines['top'].set_visible(False)
    axes[0, i].spines['right'].set_visible(False)

    axes[1, i].hist(X_scaled[feat], bins=40, color=colors_after, alpha=0.8, edgecolor='white')
    axes[1, i].set_title(f'{feat}\n(après RobustScaler)', fontsize=9)
    axes[1, i].spines['top'].set_visible(False)
    axes[1, i].spines['right'].set_visible(False)

plt.suptitle('Distributions Avant / Après RobustScaler', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 📈 9. Statistiques post-scaling

In [ ]:
key = ['throughput_mbps', 'latency_ms', 'jitter_ms', 'packet_loss_rate_pct',
       'risk_score', 'rsrp_estimated', 'sinr_estimated', 'mos_proxy',
       'prb_utilization_proxy', 'instability_score']

X_scaled[key].describe().round(3)

## 💾 10. Sauvegarde & Téléchargement

In [ ]:
import os
output_dir = '/content/outputs'
os.makedirs(output_dir, exist_ok=True)

# Dataset X préparé
X_scaled.to_csv(f'{output_dir}/X_prepared.csv', index=False)

# Target encodée
y_df = pd.DataFrame({'rsrp_category_encoded': y, 'rsrp_category_label': df[target_col].values})
y_df.to_csv(f'{output_dir}/y_target.csv', index=False)

# Dataset complet
full = X_scaled.copy()
full['rsrp_category_encoded'] = y
full['rsrp_category_label'] = df[target_col].values
full.to_csv(f'{output_dir}/qos_prepared_full.csv', index=False)

# Objets pipeline
joblib.dump(knn_imputer, f'{output_dir}/knn_imputer.pkl')
joblib.dump(scaler,      f'{output_dir}/robust_scaler.pkl')
joblib.dump(le,          f'{output_dir}/label_encoder.pkl')

print('Fichiers générés :')
for f in os.listdir(output_dir):
    size = os.path.getsize(f'{output_dir}/{f}')
    print(f'  ✅ {f:35s} ({size/1024:.1f} KB)')

## ✅ Résumé Final

In [ ]:
print('=' * 60)
print('  RÉSUMÉ FINAL')
print('=' * 60)
print(f'  Lignes           : {X_scaled.shape[0]}')
print(f'  Features (X)     : {X_scaled.shape[1]}')
print(f'  Target classes   : {len(le.classes_)} → {list(le.classes_)}')
print(f'  Imputer          : KNNImputer (k=5, weights=distance)')
print(f'  Scaler           : RobustScaler (IQR 10–90)')
print(f'  NaN dans X final : {X_scaled.isnull().sum().sum()}')
print('=' * 60)
print('\n✅ Pipeline terminé avec succès !')

## 📦 Cellule 1 — Installation des dépendances

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 1 — Installation
# Exécuter une seule fois, puis redémarrer le kernel si nécessaire
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

packages = ['ping3', 'icmplib', 'psutil', 'requests', 'ipywidgets', 'jupyterlab']
for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f'{status} {pkg}')

print()
print('⚠️  Si premier lancement → redémarrer le kernel (Kernel > Restart)')
print('   puis exécuter les cellules 1 → 2 → 3')


## ⚙️ Cellule 2 — Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 2 — CONFIGURATION
# Seuils selon Cisco QoS Design Guide + ITU-T G.114 + 3GPP TS 36
# ═══════════════════════════════════════════════════════════════
import os

# ── Réseau
TARGET_HOST  = '8.8.8.8'
TARGET_HOST2 = '1.1.1.1'
PING_COUNT   = 5
MAX_HOPS     = 10
INTERVAL_SEC = 10

# ── Chemins CSV
OUTPUT_DIR  = os.path.join(os.getcwd(), 'qos_output')
CSV_LIVE    = os.path.join(OUTPUT_DIR, 'monitoring_live.csv')
CSV_ALERTS  = os.path.join(OUTPUT_DIR, 'alerts_log.csv')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Webhook
WEBHOOK_URL = None

# ── Seuils Cisco / ITU-T / 3GPP
THRESHOLDS = {
    # Latence — ITU-T G.114
    'latency_ms':                (150,   200,   False),
    'mean_latency_ms':           (150,   200,   False),
    'max_latency_ms':            (200,   400,   False),
    'std_latency_ms':            (20,    50,    False),
    # Jitter — ITU-T G.114
    'jitter_ms':                 (30,    50,    False),
    'latency_spread':            (20,    50,    False),
    'latency_trend':             (50,    100,   False),
    # Perte paquets — Cisco QoS
    'packet_loss_rate_pct':      (1.0,   5.0,   False),
    'risk_score':                (0.4,   0.6,   False),
    'instability_score':         (5.0,   15.0,  False),
    # Bande passante — Cisco (seuils utilisation réseau)
    'bandwidth_utilization_pct': (50.0,  75.0,  False),
    'network_load':              (0.5,   0.75,  False),
    # Charge système
    'queue_length':              (70,    90,    False),
    'buffer_occupancy_pct':      (75,    90,    False),
    'prb_utilization_proxy':     (50.0,  75.0,  False),
    # Hops
    'hops_mean':                 (50,    100,   False),
    'hops_max':                  (150,   300,   False),
    'hops_std':                  (30,    60,    False),
    'hops_range':                (100,   200,   False),
    # ── throughput et available_bandwidth SUPPRIMÉS
    #    (psutil mesure le trafic local, pas la vraie bande passante)
    # MOS — ITU-T P.800
    'mos_proxy':                 (3.6,   3.1,   True),
    # Signal LTE — 3GPP TS 36.133
    'rsrp_estimated':            (-80,   -100,  True),
    'sinr_estimated':            (0,     -3,    True),
    'cqi_estimated':             (7,     4,     True),
    # Efficacité bande passante
    'bandwidth_efficiency':      (0.3,   0.1,   True),
}

BINARY_FLAGS = {
    'spike':               'Pic de latence (ITU-T G.114)',
    'ho_failure_proxy':    'Échec handover (3GPP TS 36)',
    'coverage_hole_proxy': 'Zone couverture faible (RSRP < -100 dBm)',
    'performance_degraded':'Dégradation performance (Cisco QoS)',
}

print('✅ Configuration chargée — seuils Cisco/ITU-T/3GPP')
print(f'   Cible      : {TARGET_HOST} / {TARGET_HOST2}')
print(f'   Intervalle : {INTERVAL_SEC}s')
print(f'   CSV live   : {CSV_LIVE}')
print(f'   CSV alertes: {CSV_ALERTS}')
print(f'   Webhook    : {WEBHOOK_URL or "désactivé"}')
print(f'   Seuils     : {len(THRESHOLDS)} features continues + {len(BINARY_FLAGS)} binaires')

## 🚀 Cellule 3 — Agent 1 + Agent 2 + Dashboard temps réel

In [ ]:
# ── Vérification des dépendances ───────────────────────────────
_required = ['CSV_LIVE', 'CSV_ALERTS', 'TARGET_HOST', 'TARGET_HOST2',
             'PING_COUNT', 'MAX_HOPS', 'INTERVAL_SEC',
             'THRESHOLDS', 'BINARY_FLAGS', 'WEBHOOK_URL']

_missing = [v for v in _required if v not in dir()]
if _missing:
    raise RuntimeError(
        f"❌ Variables manquantes : {_missing}\n"
        f"   → Exécute d'abord la Cellule 2 (Configuration) !"
    )
print('✅ Dépendances OK — chargement des agents...')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 3 — AGENT 1 + AGENT 2 + DASHBOARD
# Jupyter Lab / Notebook / VS Code — sans widgets
# ═══════════════════════════════════════════════════════════════
import psutil, time, os, csv, json as _json, math, statistics
import requests, threading, queue
from ping3 import ping
from icmplib import traceroute
from datetime import datetime
from IPython.display import clear_output

# ────────────────────────────────────────────────────────────────
# ÉTAT PARTAGÉ + FILE INTER-AGENTS
# ────────────────────────────────────────────────────────────────
state = {
    'running':       False,
    'total':         0,
    'alerts_count':  0,
    'last_row':      None,
    'last_analysis': None,
    'a1_log':        [],   # historique logs Agent 1
    'a2_log':        [],   # historique logs Agent 2
    'api_status':    '—',
}
data_queue    = queue.Queue(maxsize=5)
alert_history = []

# ────────────────────────────────────────────────────────────────
# AGENT 1 — CAPTURE RÉSEAU (54 features)
# ────────────────────────────────────────────────────────────────
def _save_csv(row, path):
    exists = os.path.isfile(path)
    with open(path, 'a', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=row.keys(), delimiter=';')
        if not exists:
            w.writeheader()
        w.writerow(row)

def _a1_log(msg):
    ts = datetime.now().strftime('%H:%M:%S')
    state['a1_log'].insert(0, f'[{ts}] {msg}')
    state['a1_log'] = state['a1_log'][:20]

def agent1_capture():
    ts  = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    now = datetime.now()
    row = {'timestamp': ts}

    # 1. LATENCE
    pings, host = [], TARGET_HOST
    for _ in range(PING_COUNT):
        try:
            r = ping(TARGET_HOST, timeout=2, unit='ms')
            if r: pings.append(r)
        except: pass
    if not pings:
        host = TARGET_HOST2
        for _ in range(PING_COUNT):
            try:
                r = ping(TARGET_HOST2, timeout=2, unit='ms')
                if r: pings.append(r)
            except: pass

    row['latency_ms']           = round(sum(pings)/len(pings), 4) if pings else -1
    row['mean_latency_ms']      = row['latency_ms']
    row['min_latency_ms']       = round(min(pings), 4)            if pings else -1
    row['max_latency_ms']       = round(max(pings), 4)            if pings else -1
    row['std_latency_ms']       = round(statistics.stdev(pings), 4) if len(pings)>1 else 0.0
    row['jitter_ms']            = round(row['max_latency_ms'] - row['min_latency_ms'], 4) if pings else -1
    row['latency_spread']       = row['std_latency_ms']
    row['packet_loss_rate_pct'] = round((PING_COUNT - len(pings)) / PING_COUNT * 100, 2)
    row['spike']                = 1 if row['latency_ms'] > 200 else 0
    prev = float(state['last_row']['latency_ms']) if state['last_row'] else row['latency_ms']
    row['latency_trend']        = round(row['latency_ms'] - prev, 4)

    # 2. BANDE PASSANTE
    net1 = psutil.net_io_counters(); time.sleep(1); net2 = psutil.net_io_counters()
    s = (net2.bytes_sent - net1.bytes_sent) * 8
    r = (net2.bytes_recv - net1.bytes_recv) * 8
    row['throughput_mbps']            = round(r / 1e6, 4)
    row['available_bandwidth_mbps']   = round((s + r) / 1e6, 4)
    row['bandwidth_utilization_pct']  = round(min(r / 1e8 * 100, 100), 4)
    row['bandwidth_efficiency']       = round(r / max(s + r, 1), 4)
    row['network_load']               = round(row['bandwidth_utilization_pct'] / 100, 4)

    # 3. QUALITÉ RÉSEAU
    drops = (net2.dropin + net2.dropout) - (net1.dropin + net1.dropout)
    pkts  = max(net2.packets_recv - net1.packets_recv, 1)
    row['packet_loss_rate_pct'] = round(drops / pkts * 100, 4)
    row['instability_score']    = round(row['jitter_ms'] * row['packet_loss_rate_pct'] / 100, 4)
    row['risk_score']           = round((row['latency_ms']/300 + row['packet_loss_rate_pct']/100) / 2, 4)

    # 4. CHARGE SYSTÈME
    row['buffer_occupancy_pct']  = round(psutil.virtual_memory().percent, 4)
    row['queue_length']          = round(psutil.cpu_percent(interval=0.2), 4)
    row['congestion_level']      = 0
    row['prb_utilization_proxy'] = row['bandwidth_utilization_pct']

    # 5. HOPS
    hl = []
    try:
        hops = traceroute(host, max_hops=MAX_HOPS, timeout=1)
        hl = [h.avg_rtt for h in hops if h.avg_rtt > 0]
    except: pass
    for i in range(1, 11):
        row[f'hop_{i}'] = round(hl[i-1], 4) if i <= len(hl) else 0.0
    row['hops_mean']  = round(sum(hl)/len(hl), 4) if hl else 0.0
    row['hops_max']   = round(max(hl), 4)          if hl else 0.0
    row['hops_min']   = round(min(hl), 4)          if hl else 0.0
    row['hops_std']   = round(statistics.stdev(hl), 4) if len(hl) > 1 else 0.0
    row['hops_range'] = round(row['hops_max'] - row['hops_min'], 4)

    # 6. SIGNAL ESTIMÉ
    row['rsrp_estimated'] = round(-70 - row['latency_ms'] * 0.1, 4)
    row['sinr_estimated'] = round(20 - row['packet_loss_rate_pct'] * 2, 4)
    row['cqi_estimated']  = max(1, min(15, int(15 - row['packet_loss_rate_pct'] / 7)))
    row['mos_proxy']      = round(max(1, 4.5 - row['packet_loss_rate_pct']*0.1 - row['latency_ms']*0.005), 4)

    # 7. FLAGS BINAIRES
    row['ho_failure_proxy']     = 1 if row['packet_loss_rate_pct'] > 5 else 0
    row['coverage_hole_proxy']  = 1 if row['rsrp_estimated'] < -100    else 0
    row['performance_degraded'] = 1 if (row['latency_ms'] > 150 or row['packet_loss_rate_pct'] > 3) else 0

    # 8. CATÉGORIE RSRP
    rsrp = row['rsrp_estimated']
    row['rsrp_category'] = ('Bon'          if rsrp >= -80  else
                            'Mauvais'      if rsrp >= -90  else
                            'Faible'       if rsrp >= -100 else
                            'Très mauvais')

    # 9. TEMPOREL
    row['hour']                   = now.hour
    row['minute']                 = now.minute
    row['dayofweek']              = now.weekday()
    row['hour_sin']               = round(math.sin(2*math.pi*now.hour/24), 4)
    row['hour_cos']               = round(math.cos(2*math.pi*now.hour/24), 4)
    row['minute_sin']             = round(math.sin(2*math.pi*now.minute/60), 4)
    row['minute_cos']             = round(math.cos(2*math.pi*now.minute/60), 4)
    row['dayofweek_sin']          = round(math.sin(2*math.pi*now.weekday()/7), 4)
    row['dayofweek_cos']          = round(math.cos(2*math.pi*now.weekday()/7), 4)
    row['peak_offpeak_indicator'] = 1 if 8 <= now.hour <= 22 else 0
    return row

def agent1_loop():
    _a1_log('démarré')
    while state['running']:
        state['total'] += 1
        n = state['total']
        _a1_log(f'⏱  capture #{n} en cours...')
        try:
            row = agent1_capture()
            state['last_row'] = row
            _save_csv(row, CSV_LIVE)
            try: data_queue.put_nowait(row)
            except: pass
            _a1_log(f'✅ #{n} — {len(row)} features — latence {row["latency_ms"]} ms')
        except Exception as e:
            _a1_log(f'❌ #{n} erreur: {e}')
        for _ in range(INTERVAL_SEC):
            if not state['running']: break
            time.sleep(1)
    _a1_log(f'⏹  arrêté après {state["total"]} captures')

# ────────────────────────────────────────────────────────────────
# AGENT 2 — CLASSIFICATION & ALERTES
# ────────────────────────────────────────────────────────────────
def _a2_log(msg):
    ts = datetime.now().strftime('%H:%M:%S')
    state['a2_log'].insert(0, f'[{ts}] {msg}')
    state['a2_log'] = state['a2_log'][:20]

def agent2_classify(row):
    alerts = []
    for metric, (warn, crit, inv) in THRESHOLDS.items():
        val = row.get(metric)
        if val is None or val == -1: continue
        v = float(val)
        level = ('CRITICAL' if (v <= crit if inv else v >= crit) else
                 'WARNING'  if (v <= warn if inv else v >= warn) else 'OK')
        if level != 'OK':
            alerts.append({
                'metric':    metric,
                'value':     round(v, 4),
                'threshold': crit if level == 'CRITICAL' else warn,
                'level':     level,
            })
    for metric, desc in BINARY_FLAGS.items():
        if int(float(row.get(metric, 0))) == 1:
            alerts.append({'metric': metric, 'value': 1,
                           'description': desc, 'level': 'CRITICAL'})
    has_crit = any(a['level'] == 'CRITICAL' for a in alerts)
    return {
        'alerts':       alerts,
        'severity':     'CRITICAL' if has_crit else ('WARNING' if alerts else 'OK'),
        'has_critical': has_crit,
        'rsrp_class':   row.get('rsrp_category', '?'),
        'n_critical':   sum(1 for a in alerts if a['level'] == 'CRITICAL'),
        'n_warning':    sum(1 for a in alerts if a['level'] == 'WARNING'),
    }

def build_payload(row, analysis):
    return {
        'agent':     'QoS-Agent2-Classifier',
        'timestamp':  row.get('timestamp', ''),
        'target':     TARGET_HOST,
        'severity':   analysis['severity'],
        'rsrp_class': analysis['rsrp_class'],
        'n_critical': analysis['n_critical'],
        'n_warning':  analysis['n_warning'],
        'alerts':     analysis['alerts'],
        'metrics':    {k: row.get(k) for k in [
            'latency_ms','mean_latency_ms','max_latency_ms','jitter_ms','latency_trend',
            'packet_loss_rate_pct','risk_score','instability_score',
            'throughput_mbps','bandwidth_utilization_pct','mos_proxy',
            'rsrp_estimated','sinr_estimated','cqi_estimated','hops_mean',
            'buffer_occupancy_pct','queue_length','spike','ho_failure_proxy',
            'coverage_hole_proxy','performance_degraded','rsrp_category',
        ]}
    }

def send_webhook(payload):
    if not WEBHOOK_URL: return 'désactivé'
    try:
        r = requests.post(WEBHOOK_URL, json=payload, timeout=5,
                          headers={'Content-Type': 'application/json'})
        r.raise_for_status()
        return f'HTTP {r.status_code} ✅'
    except Exception as e:
        return f'Erreur: {e}'

def agent2_loop():
    _a2_log('démarré — en écoute')
    while state['running']:
        try: row = data_queue.get(timeout=2)
        except: continue
        try:
            analysis = agent2_classify(row)
            state['last_analysis'] = analysis

            # ── CORRECTIF : transmettre (row, analysis) à DSO3
            try:
                dso3_input_queue.put_nowait((row, analysis))
            except queue.Full:
                pass

            if analysis['severity'] != 'OK':
                state['alerts_count'] += 1
                # Sauvegarde CSV alertes
                exists = os.path.isfile(CSV_ALERTS)
                with open(CSV_ALERTS, 'a', newline='', encoding='utf-8') as f:
                    fw = csv.DictWriter(f, delimiter=';', fieldnames=[
                        'timestamp','severity','rsrp_class',
                        'metric','value','threshold','level','description'])
                    if not exists: fw.writeheader()
                    for a in analysis['alerts']:
                        fw.writerow({
                            'timestamp':   row['timestamp'],
                            'severity':    analysis['severity'],
                            'rsrp_class':  analysis['rsrp_class'],
                            'metric':      a.get('metric',''),
                            'value':       a.get('value',''),
                            'threshold':   a.get('threshold',''),
                            'level':       a['level'],
                            'description': a.get('description', a.get('metric','')),
                        })
                if WEBHOOK_URL:
                    state['api_status'] = send_webhook(build_payload(row, analysis))
                _a2_log(f'🚨 {analysis["severity"]} — {analysis["n_critical"]}C {analysis["n_warning"]}W alertes')
            else:
                _a2_log(f'✅ OK — RSRP: {analysis["rsrp_class"]}')

        except Exception as e:
            _a2_log(f'❌ erreur: {e}')
    _a2_log(f'⏹  arrêté — {state["alerts_count"]} alertes déclenchées')

# ────────────────────────────────────────────────────────────────
# DASHBOARD — affichage séparé Agent 1 / Agent 2
# ────────────────────────────────────────────────────────────────
GROUPS_A1 = {
    '⏱️  Latence':       ['latency_ms','mean_latency_ms','min_latency_ms','max_latency_ms',
                          'std_latency_ms','jitter_ms','latency_spread','latency_trend','spike'],
    '📶 Bande passante': ['throughput_mbps','available_bandwidth_mbps',
                          'bandwidth_utilization_pct','bandwidth_efficiency','network_load'],
    '📉 Qualité réseau': ['packet_loss_rate_pct','instability_score','risk_score'],
    '🔧 Charge système': ['queue_length','buffer_occupancy_pct','prb_utilization_proxy'],
    '📡 Signal estimé':  ['rsrp_estimated','sinr_estimated','cqi_estimated','mos_proxy','rsrp_category'],
    '🛤️  Hops':          ['hops_mean','hops_max','hops_min','hops_std','hops_range',
                          'hop_1','hop_2','hop_3','hop_4','hop_5',
                          'hop_6','hop_7','hop_8','hop_9','hop_10'],
    '🕐 Temporel':       ['hour','minute','dayofweek','peak_offpeak_indicator',
                          'hour_sin','hour_cos','minute_sin','minute_cos',
                          'dayofweek_sin','dayofweek_cos'],
    '🚨 Flags binaires': ['spike','ho_failure_proxy','coverage_hole_proxy','performance_degraded'],
}

def _flag_note(metric, val):
    t = THRESHOLDS.get(metric)
    if not t: return ''
    warn, crit, inv = t
    v = float(val)
    if inv:
        if v <= crit: return '  🔴 CRITIQUE'
        if v <= warn: return '  🟡 AVERT.'
    else:
        if v >= crit: return '  🔴 CRITIQUE'
        if v >= warn: return '  🟡 AVERT.'
    return ''

def render():
    row = state.get('last_row')
    ana = state.get('last_analysis')
    n   = state.get('total', 0)
    clear_output(wait=True)

    # ══════════════════════════════════════════════════════
    # HEADER GLOBAL
    # ══════════════════════════════════════════════════════
    sev   = ana['severity'] if ana else '—'
    emoji = '🔴' if sev=='CRITICAL' else ('🟡' if sev=='WARNING' else '🟢')
    print('╔' + '═'*63 + '╗')
    print(f'║  🛰️  QoS Real-Time Monitor — DSO1' + ' '*29 + '║')
    print(f'║  Capture #{n:<6} · {datetime.now().strftime("%H:%M:%S")}' + ' '*36 + '║')
    print(f'║  Statut : {emoji} {sev:<10} · Alertes totales : {state["alerts_count"]:<6}' + ' '*14 + '║')
    print('╚' + '═'*63 + '╝')

    # ══════════════════════════════════════════════════════
    # AGENT 1 — CAPTURE
    # ══════════════════════════════════════════════════════
    print()
    print('┌─────────────────────────────────────────────────────────────┐')
    print('│  🤖  AGENT 1 — Capture réseau (54 features)                 │')
    print('└─────────────────────────────────────────────────────────────┘')

    if not row:
        print('  ⏳ En attente de la première capture...')
    else:
        for grp, feats in GROUPS_A1.items():
            visible = [f for f in feats if f in row]
            if not visible: continue
            print(f'\n  {grp}')
            print(f'  {"─"*58}')
            print(f'  {"Feature":<36} {"Valeur":>10}  {"Statut"}')
            print(f'  {"─"*58}')
            for feat in visible:
                val  = row[feat]
                note = ''
                if feat in BINARY_FLAGS:
                    note = '🔴 ACTIF' if int(float(val))==1 else '✅ OK'
                elif feat in THRESHOLDS:
                    n_raw = _flag_note(feat, val)
                    note  = n_raw.strip() if n_raw else '✅ OK'
                print(f'  {feat:<36} {str(val):>10}  {note}')

    # Logs Agent 1
    print(f'\n  📋 Logs Agent 1 :')
    for line in state['a1_log'][:5]:
        print(f'     {line}')

    print(f'\n  💾 CSV live → {CSV_LIVE}')

    # ══════════════════════════════════════════════════════
    # AGENT 2 — CLASSIFICATION
    # ══════════════════════════════════════════════════════
    print()
    print('┌─────────────────────────────────────────────────────────────┐')
    print('│  🔍  AGENT 2 — Classification & Alertes                     │')
    print('└─────────────────────────────────────────────────────────────┘')

    if not ana:
        print('  ⏳ En attente de la première classification...')
    else:
        sev_a2   = ana['severity']
        emoji_a2 = '🔴' if sev_a2=='CRITICAL' else ('🟡' if sev_a2=='WARNING' else '🟢')
        print(f'\n  Sévérité   : {emoji_a2} {sev_a2}')
        print(f'  RSRP class : {ana["rsrp_class"]}')
        print(f'  Critiques  : {ana["n_critical"]}   Avertissements : {ana["n_warning"]}')

        if ana['alerts']:
            print(f'\n  {"─"*58}')
            print(f'  {"Metric":<32} {"Valeur":>8}  {"Seuil":>8}  {"Niveau"}')
            print(f'  {"─"*58}')
            for a in ana['alerts']:
                icon  = '🔴' if a['level']=='CRITICAL' else '🟡'
                desc  = a.get('description', a['metric'])
                val   = str(a.get('value',''))
                thr   = str(a.get('threshold',''))
                print(f'  {icon} {desc:<30} {val:>8}  {thr:>8}  {a["level"]}')
        else:
            print('\n  ✅ Aucune alerte — tous les indicateurs dans les seuils')

    # Logs Agent 2
    print(f'\n  📋 Logs Agent 2 :')
    for line in state['a2_log'][:5]:
        print(f'     {line}')

    print(f'\n  💾 CSV alertes → {CSV_ALERTS}')
    if WEBHOOK_URL:
        print(f'  📤 Webhook     → {state.get("api_status","—")}')

    # ══════════════════════════════════════════════════════
    # JOURNAL ALERTES (5 dernières)
    # ══════════════════════════════════════════════════════
    if alert_history:
        print()
        print('┌─────────────────────────────────────────────────────────────┐')
        print('│  🚨  Journal alertes                                         │')
        print('└─────────────────────────────────────────────────────────────┘')
        for line in alert_history[:5]:
            print(f'  {line}')
        if len(alert_history) > 5:
            print(f'  ... +{len(alert_history)-5} alertes dans le journal complet')

    print()
    print('  ▶ on_start()  ■ on_stop()  ↺ on_reset()  🔄 render()')

    # Mise à jour journal alertes
    if ana and ana['alerts'] and row:
        ts  = row['timestamp']
        sev = ana['severity']
        for a in ana['alerts']:
            desc  = a.get('description', f'{a["metric"]}={a["value"]} seuil={a.get("threshold","")}')
            entry = f'[{ts}] [{sev}] {desc}'
            if entry not in alert_history:
                alert_history.insert(0, entry)
        alert_history[:] = alert_history[:200]

def dashboard_loop():
    while state['running']:
        try: render()
        except: pass
        time.sleep(INTERVAL_SEC)

# ────────────────────────────────────────────────────────────────
# COMMANDES
# ────────────────────────────────────────────────────────────────
def on_start():
    if state['running']:
        print('⚠️  Déjà en cours — on_stop() pour arrêter')
        return
    state.update({
        'running': True, 'total': 0, 'alerts_count': 0,
        'last_row': None, 'last_analysis': None,
        'a1_log': [], 'a2_log': [], 'api_status': '—',
    })
    alert_history.clear()
    threading.Thread(target=agent1_loop,    daemon=True).start()
    threading.Thread(target=agent2_loop,    daemon=True).start()
    threading.Thread(target=dashboard_loop, daemon=True).start()
    print(f'🟢 Agents démarrés — intervalle {INTERVAL_SEC}s — cible {TARGET_HOST}')
    print('   Le dashboard se rafraîchit automatiquement.')
    print('   Pour affichage immédiat → render()')
'''
def on_stop():
    state['running'] = False
    print(f'⏹  Arrêté après {state["total"]} captures · {state["alerts_count"]} alertes')

def on_reset():
    state['running'] = False
    alert_history.clear()
    state.update({'total':0,'alerts_count':0,'last_row':None,'last_analysis':None,
                  'a1_log':[],'a2_log':[],'api_status':'—'})
    print('↺ Réinitialisé — prêt')
'''
print('✅ Agents définis')
print('▶  on_start()  →  démarrer')
print('■  on_stop()   →  arrêter')
print('↺  on_reset()  →  réinitialiser')
print('🔄  render()   →  affichage immédiat')

In [ ]:
on_start()

In [60]:
#on_stop()

### DSO 3 — PRÉDICTION DES RISQUES EN TEMPS RÉEL

In [ ]:
# ═══════════════════════════════════════════════════════════════
# DSO 3 — PRÉDICTION DES RISQUES EN TEMPS RÉEL
# Pipeline : DSO1 (Agent1+Agent2) → DSO3 → valise → DSO4
# Jupyter Lab / Notebook / VS Code — sans widgets
# ═══════════════════════════════════════════════════════════════
 
# ──────────────────────────────────────────────────────────────
# CELLULE 1 — CONFIGURATION & CONSTANTES DSO 3
# ──────────────────────────────────────────────────────────────
 
import os
 
# ── Fichiers de sortie DSO 3
CSV_PREDICTIONS  = 'dso3_predictions.csv'
CSV_VALISE       = 'dso3_valise.csv'          # payload complet transmis à DSO4
 
# ── Horizons de prédiction
HORIZONS_SEC     = [20, 60, 300]              # +20s, +60s, +5min
 
# ── Seuils risk score (0–100)
RISK_THRESHOLDS  = {
    'LOW':      (0,   30),
    'MEDIUM':   (30,  60),
    'HIGH':     (60,  80),
    'CRITICAL': (80, 100),
}
 
# ── Features continues utilisées par le séquenceur temporel
SEQ_FEATURES = [
    'latency_ms', 'jitter_ms', 'packet_loss_rate_pct',
    'throughput_mbps', 'risk_score', 'instability_score',
    'rsrp_estimated', 'sinr_estimated', 'mos_proxy',
    'buffer_occupancy_pct', 'queue_length',
    'bandwidth_utilization_pct', 'hops_mean',
]
 
# ── Features tabulaires pour le classifieur XGB-like
TAB_FEATURES = SEQ_FEATURES + [
    'spike', 'ho_failure_proxy', 'coverage_hole_proxy', 'performance_degraded',
    'peak_offpeak_indicator', 'hour_sin', 'hour_cos',
    'dayofweek_sin', 'dayofweek_cos',
    # injections Agent 2
    'severity_encoded', 'n_critical', 'n_warning',
    # delta features (calculées dynamiquement)
    'delta_latency', 'delta_jitter', 'delta_packet_loss',
    'delta_throughput', 'delta_risk_score',
]
 
# ── Longueur fenêtre glissante (nombre de captures)
WINDOW_SIZE     = 15
MIN_WINDOW_PRED =  5    # prédire dès 5 captures disponibles
 
# ── Poids ensemble (LSTM-like : seq_model, XGB-like : tab_model)
W_SEQ = 0.55
W_TAB = 0.45
 
# ── Webhook DSO4 (optionnel, laisser vide pour désactiver)
WEBHOOK_DSO4_URL = os.environ.get('WEBHOOK_DSO4_URL', '')
 
print('✅ DSO3 — Configuration chargée')
print(f'   Horizons  : {HORIZONS_SEC} s')
print(f'   Fenêtre   : {WINDOW_SIZE} captures')
print(f'   Seq feat  : {len(SEQ_FEATURES)}  |  Tab feat : {len(TAB_FEATURES)}')
print(f'   CSV prédictions → {CSV_PREDICTIONS}')
print(f'   CSV valise      → {CSV_VALISE}')
 

### IMPORTS & DÉPENDANCES DSO 3

In [ ]:
# ──────────────────────────────────────────────────────────────
# CELLULE 2 — IMPORTS & DÉPENDANCES DSO 3
# ──────────────────────────────────────────────────────────────

import numpy as np
import math, csv, json, time, threading, queue
from collections import deque
from datetime import datetime
import requests

# ── Imports ML (fallback automatique si bibliothèque absente)
try:
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.ensemble import GradientBoostingClassifier, IsolationForest
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    _sklearn_ok = True
    print('✅ scikit-learn disponible')
except ImportError:
    _sklearn_ok = False
    print('⚠️  scikit-learn absent — modèle statistique de secours activé')

try:
    import xgboost as xgb
    _xgb_ok = True
    print('✅ XGBoost disponible')
except ImportError:
    _xgb_ok = False
    print('⚠️  XGBoost absent — GradientBoosting sklearn utilisé')

try:
    import torch, torch.nn as nn
    _torch_ok = True
    print('✅ PyTorch disponible — modèle LSTM activé')
except ImportError:
    _torch_ok = False
    print('⚠️  PyTorch absent — prédiction séquentielle statistique activée')

print('\n✅ Imports DSO3 terminés')


### MODÈLE SÉQUENTIEL (LSTM PyTorch OU FALLBACK)

In [ ]:
# ──────────────────────────────────────────────────────────────
# CELLULE 3 — MODÈLE SÉQUENTIEL (LSTM PyTorch OU FALLBACK)
# ──────────────────────────────────────────────────────────────

if _torch_ok:
    class LSTMRiskModel(nn.Module):
        """LSTM bi-directionnel pour prédiction série temporelle réseau."""
        def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
            super().__init__()
            self.lstm = nn.LSTM(
                input_size=input_size,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=False,
                dropout=dropout if num_layers > 1 else 0.0,
            )
            self.attention = nn.Sequential(
                nn.Linear(hidden_size, 32),
                nn.Tanh(),
                nn.Linear(32, 1),
                nn.Softmax(dim=1),
            )
            self.head = nn.Sequential(
                nn.Linear(hidden_size, 32),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(32, 1),
                nn.Sigmoid(),
            )

        def forward(self, x):
            # x : (batch, seq_len, input_size)
            out, _ = self.lstm(x)            # (batch, seq, hidden)
            attn_w = self.attention(out)     # (batch, seq, 1)
            ctx    = (out * attn_w).sum(1)   # (batch, hidden) — contexte pondéré
            return self.head(ctx).squeeze(-1)

    def build_lstm_model(input_size):
        model = LSTMRiskModel(input_size=input_size)
        print(f'   LSTM créé — {sum(p.numel() for p in model.parameters()):,} paramètres')
        return model

else:
    # ── Fallback statistique : EWMA + tendance linéaire
    class StatSeqModel:
        """Prédicteur statistique EWMA quand PyTorch est absent."""
        def __init__(self, alpha=0.3):
            self.alpha = alpha

        def predict_score(self, seq_array):
            """seq_array : np.array (seq_len, n_feat) — retourne score 0-1."""
            if len(seq_array) < 2:
                return 0.3
            # Canal 0 = latency_ms normalisé, canal 4 = risk_score normalisé
            lat = seq_array[:, 0]
            rsk = seq_array[:, 4] if seq_array.shape[1] > 4 else lat
            # EWMA
            ewma = lat[0]
            for v in lat[1:]:
                ewma = self.alpha * v + (1 - self.alpha) * ewma
            # Tendance linéaire (pente normalisée)
            n   = len(lat)
            x   = np.arange(n)
            trend = np.polyfit(x, lat, 1)[0]  # pente
            score = np.clip(ewma * 0.6 + rsk[-1] * 0.3 + np.clip(trend * 5, 0, 1) * 0.1, 0, 1)
            return float(score)

    def build_lstm_model(input_size):
        print(f'   Fallback StatSeqModel activé (PyTorch absent)')
        return StatSeqModel()

print('✅ Modèle séquentiel défini')


###  MODÈLE TABULAIRE (XGBoost OU FALLBACK SKLEARN)

In [ ]:
# ──────────────────────────────────────────────────────────────
# CELLULE 4 — MODÈLE TABULAIRE (XGBoost OU FALLBACK SKLEARN)
# ──────────────────────────────────────────────────────────────

class TabRiskClassifier:
    """
    Classifieur tabulaire : XGBoost si disponible, sinon GradientBoosting sklearn.
    Entraîné en ligne (online learning partiel via warm_start).
    Retourne une probabilité de risque élevé (classe ≥ HIGH).
    """
    SEV_MAP = {'OK': 0, 'WARNING': 1, 'CRITICAL': 2}

    def __init__(self):
        self.scaler   = MinMaxScaler() if _sklearn_ok else None
        self.fitted   = False
        self.buffer_X = []   # features accumulées avant premier fit
        self.buffer_y = []   # labels pseudo (générés par seuils Agent2)
        self.MIN_FIT  = 30   # captures mini avant premier entraînement

        if _xgb_ok:
            self.model = xgb.XGBClassifier(
                n_estimators=100, max_depth=4, learning_rate=0.1,
                subsample=0.8, colsample_bytree=0.8,
                use_label_encoder=False, eval_metric='logloss',
                verbosity=0,
            )
            self.backend = 'xgboost'
        elif _sklearn_ok:
            self.model = GradientBoostingClassifier(
                n_estimators=80, max_depth=3, learning_rate=0.1,
                subsample=0.8, warm_start=True,
            )
            self.backend = 'sklearn_gb'
        else:
            self.model   = None
            self.backend = 'heuristic'

        print(f'   TabClassifier backend : {self.backend}')

    def _extract(self, row, analysis):
        """Construit le vecteur tabulaire à partir d'un row + analysis Agent2."""
        vec = {}
        for f in TAB_FEATURES:
            if f == 'severity_encoded':
                vec[f] = self.SEV_MAP.get(analysis.get('severity', 'OK'), 0)
            elif f == 'n_critical':
                vec[f] = analysis.get('n_critical', 0)
            elif f == 'n_warning':
                vec[f] = analysis.get('n_warning', 0)
            elif f.startswith('delta_'):
                vec[f] = row.get(f, 0.0)   # pré-calculés par FeatureEngineer
            else:
                vec[f] = float(row.get(f, 0) or 0)
        return [vec[f] for f in TAB_FEATURES]

    def _pseudo_label(self, analysis):
        """Label pseudo supervisé dérivé de l'Agent2 (0=OK/MEDIUM, 1=HIGH/CRITICAL)."""
        sev = analysis.get('severity', 'OK')
        nc  = analysis.get('n_critical', 0)
        return 1 if sev == 'CRITICAL' or nc >= 2 else 0

    def update(self, row, analysis):
        """Ajoute un exemple au buffer et re-entraîne si seuil atteint."""
        x = self._extract(row, analysis)
        y = self._pseudo_label(analysis)
        self.buffer_X.append(x)
        self.buffer_y.append(y)

        if len(self.buffer_X) >= self.MIN_FIT and self.model is not None:
            X = np.array(self.buffer_X)
            if self.scaler and not self.fitted:
                X = self.scaler.fit_transform(X)
            elif self.scaler:
                X = self.scaler.transform(X)
            y_arr = np.array(self.buffer_y)
            if len(set(y_arr)) >= 2:  # besoin des 2 classes
                self.model.fit(X, y_arr)
                self.fitted = True

    def predict_proba(self, row, analysis):
        """Retourne probabilité de risque élevé [0.0 – 1.0]."""
        if not self.fitted or self.model is None:
            # Heuristique avant premier fit
            sev = analysis.get('severity', 'OK')
            nc  = analysis.get('n_critical', 0)
            nw  = analysis.get('n_warning', 0)
            base = {'OK': 0.1, 'WARNING': 0.4, 'CRITICAL': 0.75}.get(sev, 0.1)
            return min(1.0, base + nc * 0.08 + nw * 0.03)

        x   = np.array([self._extract(row, analysis)])
        if self.scaler:
            x = self.scaler.transform(x)
        proba = self.model.predict_proba(x)[0]
        return float(proba[1]) if len(proba) > 1 else float(proba[0])

print('✅ Classifieur tabulaire défini')

### FEATURE ENGINEER (DELTA + ROLLING + NORMALISATION)

In [ ]:
# ──────────────────────────────────────────────────────────────
# CELLULE 5 — FEATURE ENGINEER (DELTA + ROLLING + NORMALISATION)
# ──────────────────────────────────────────────────────────────

class FeatureEngineer:
    """
    Calcule les features dérivées à partir de la fenêtre glissante :
      • delta_*    : variation entre t et t-1
      • rolling_*  : moyenne mobile sur WINDOW_SIZE captures
      • seq_array  : tableau normalisé (seq_len, n_seq_feat) pour LSTM
    """
    def __init__(self, window_size=WINDOW_SIZE):
        self.window      = deque(maxlen=window_size)
        self.seq_scaler  = MinMaxScaler() if _sklearn_ok else None
        self.seq_fitted  = False

    def push(self, row, analysis):
        """Ajoute une capture et retourne row enrichi."""
        enriched = dict(row)
        sev_map  = {'OK': 0, 'WARNING': 1, 'CRITICAL': 2}
        enriched['severity_encoded'] = sev_map.get(
            analysis.get('severity', 'OK'), 0)
        enriched['n_critical'] = analysis.get('n_critical', 0)
        enriched['n_warning']  = analysis.get('n_warning',  0)

        # ── Deltas
        if self.window:
            prev = self.window[-1]
            for feat in ['latency_ms', 'jitter_ms', 'packet_loss_rate_pct',
                         'throughput_mbps', 'risk_score']:
                prev_val = float(prev.get(feat, 0) or 0)
                curr_val = float(enriched.get(feat, 0) or 0)
                enriched[f'delta_{feat}'] = round(curr_val - prev_val, 4)
        else:
            for feat in ['latency_ms', 'jitter_ms', 'packet_loss_rate_pct',
                         'throughput_mbps', 'risk_score']:
                enriched[f'delta_{feat}'] = 0.0

        # ── Rolling moyennes
        if len(self.window) >= 3:
            win_list = list(self.window)
            for feat in ['latency_ms', 'jitter_ms', 'packet_loss_rate_pct']:
                vals = [float(w.get(feat, 0) or 0) for w in win_list[-5:]]
                enriched[f'rolling_mean_{feat}'] = round(np.mean(vals), 4)
                enriched[f'rolling_std_{feat}']  = round(np.std(vals), 4)

        self.window.append(enriched)
        return enriched

    def get_seq_array(self):
        """
        Retourne np.array (seq_len, n_seq_feat) normalisé pour LSTM.
        Retourne None si fenêtre insuffisante.
        """
        if len(self.window) < MIN_WINDOW_PRED:
            return None

        raw = []
        for w in self.window:
            raw.append([float(w.get(f, 0) or 0) for f in SEQ_FEATURES])
        raw = np.array(raw)  # (seq_len, n_feat)

        if self.seq_scaler:
            if not self.seq_fitted and len(raw) >= 5:
                self.seq_scaler.fit(raw)
                self.seq_fitted = True
            if self.seq_fitted:
                raw = self.seq_scaler.transform(raw)

        return raw

    @property
    def n_seq_features(self):
        return len(SEQ_FEATURES)

print('✅ FeatureEngineer défini')



### DÉTECTEUR D'ANOMALIES (ISOLATION FOREST) 

In [ ]:
# ──────────────────────────────────────────────────────────────
# CELLULE 6 — DÉTECTEUR D'ANOMALIES (ISOLATION FOREST)
# ──────────────────────────────────────────────────────────────

class AnomalyDetector:
    """
    Isolation Forest non-supervisé : détecte des comportements
    jamais vus, indépendamment des seuils Agent2.
    """
    ANOMALY_FEATURES = [
        'latency_ms', 'jitter_ms', 'packet_loss_rate_pct',
        'throughput_mbps', 'hops_mean', 'rsrp_estimated',
        'instability_score', 'risk_score',
    ]
    MIN_FIT = 50   # captures avant premier fit

    def __init__(self):
        self.buffer  = []
        self.fitted  = False
        if _sklearn_ok:
            self.model = IsolationForest(
                n_estimators=100, contamination=0.05,
                random_state=42, n_jobs=-1,
            )
        else:
            self.model = None

    def _vec(self, row):
        return [float(row.get(f, 0) or 0) for f in self.ANOMALY_FEATURES]

    def update(self, row):
        self.buffer.append(self._vec(row))
        if len(self.buffer) >= self.MIN_FIT and self.model:
            self.model.fit(np.array(self.buffer))
            self.fitted = True

    def score(self, row):
        """
        Retourne (is_anomaly: bool, anomaly_score: float [-1, 0]).
        Plus le score est bas (proche de -1), plus c'est anormal.
        """
        if not self.fitted or not self.model:
            return False, 0.0
        x = np.array([self._vec(row)])
        pred  = self.model.predict(x)[0]      # -1 anomalie, +1 normal
        score = self.model.score_samples(x)[0]
        return bool(pred == -1), round(float(score), 4)

print('✅ AnomalyDetector défini')



### RISK SCORER (ENSEMBLE + HORIZONS) 